***
*Project:* Source Separation Neural Networks

*Author:* Jingwei Liu, [Postdoctoral Research Associate](https://www.seresearch.qmul.ac.uk/cfcs/people/jiliu/), School of EECS, Queen Mary University of London

[Centre for Digital Music (C4DM) & Centre for Fundamentals of AI and Computational Theory]
***

# <span style="background-color:darkorange; color:white; padding:2px 6px">Document 2</span> 

# Source Separation with [Conv-Tas Net](https://arxiv.org/abs/1809.07454)

*Updated:* Jan 9, 2026


In [2]:
import os
import librosa
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch import optim
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Audio
import matplotlib.pyplot as plt
torch.cuda.is_available()

True

In [2]:
print(torch.__version__)

2.9.1+cu128


In [3]:
# Constants
SAMPLING_RATE = 8000  # Adjust this based on your needs
WINDOW_SIZE = SAMPLING_RATE * 4  # 4 seconds window
HOP_SIZE = SAMPLING_RATE // 2  # 50% overlap

In [4]:
# Function to load audio files and prepare windows
def load_audio_files(mixed_dir, source_dirs):
    mixed_data = []
    source_data = []
    
    for filename in os.listdir(mixed_dir):
        if filename.endswith('.wav'):
            mixed_path = os.path.join(mixed_dir, filename)
            mixed_signal, sr_mixed = librosa.load(mixed_path, sr=SAMPLING_RATE)

            # Initialize a list for the source signals
            source_signals = []

            # Load corresponding source audio from all source directories
            for source_dir in source_dirs:
                source_path = os.path.join(source_dir, filename)  # Assuming the same filename
                if os.path.exists(source_path):
                    source_signal, sr_source = librosa.load(source_path, sr=SAMPLING_RATE)
                    source_signals.append(source_signal)

            # Ensure all source signals are the same length
            if source_signals:
                min_length = min(len(mixed_signal), *[len(src) for src in source_signals])
                mixed_signal = mixed_signal[:min_length]
                source_signals = [src[:min_length] for src in source_signals]

                # Create windows with overlap for mixed and source signals
                mixed_windows = create_windows(mixed_signal)
                source_windows = [create_windows(src) for src in source_signals]

                mixed_data.extend(mixed_windows)
                
                # Flatten the list of source windows (if there are multiple sources)
                for source_window_set in zip(*source_windows):  # iterate over each source's windows
                    source_data.append(np.stack(source_window_set))  # stack them into a multi-channel format
    
    return np.array(mixed_data), np.array(source_data)

In [5]:
# Function to create overlapping windows with Hamming window
def create_windows(signal):
    windows = []
    for start in range(0, len(signal) - WINDOW_SIZE + 1, HOP_SIZE):
        window = signal[start:start + WINDOW_SIZE]
        # Apply Hamming window
        hamming_window = np.hamming(WINDOW_SIZE)
        windowed_signal = window * hamming_window
        windows.append(windowed_signal)
    return windows

In [6]:
# Custom Dataset class for PyTorch
class AudioDataset(Dataset):
    def __init__(self, mixed_data, source_data):
        self.mixed_data = mixed_data
        self.source_data = source_data

    def __len__(self):
        return len(self.mixed_data)

    def __getitem__(self, idx):
        mixed = torch.tensor(self.mixed_data[idx], dtype=torch.float32)
        source = torch.tensor(self.source_data[idx], dtype=torch.float32)
        return mixed, source

In [7]:
# Load mixed and source audio
mixed_dir_train = 'D:\Datasets\Source_Separation_Dataset\MiniLibriMix\\train\\mix_clean'
source_dirs_train = ['D:\Datasets\Source_Separation_Dataset\MiniLibriMix\\train\s1', 'D:\Datasets\Source_Separation_Dataset\MiniLibriMix\\train\s2']  # List of source directories
mixed_data_train, source_data_train = load_audio_files(mixed_dir_train, source_dirs_train)

In [8]:
# Create dataset and dataloader
audio_dataset_train = AudioDataset(np.expand_dims(mixed_data_train, axis=1), source_data_train)
train_loader = DataLoader(audio_dataset_train, batch_size=32, shuffle=True)

# # # Example of how to access data
# for mixed, source in train_loader:
#     print(mixed.shape, source.shape)  # Should print shapes of (batch_size, 22050)

In [9]:
for i, (inputs, targets) in enumerate(train_loader):
    print(f"Batch {i+1}:")
    print("Input shape:", inputs.shape)
    print("Target shape:", targets.shape)
    test_input = inputs
    test_target = targets
    
    if i == 4:   # Stop after 5 batches (0–4)
        break

Batch 1:
Input shape: torch.Size([32, 1, 32000])
Target shape: torch.Size([32, 2, 32000])
Batch 2:
Input shape: torch.Size([32, 1, 32000])
Target shape: torch.Size([32, 2, 32000])
Batch 3:
Input shape: torch.Size([32, 1, 32000])
Target shape: torch.Size([32, 2, 32000])
Batch 4:
Input shape: torch.Size([32, 1, 32000])
Target shape: torch.Size([32, 2, 32000])
Batch 5:
Input shape: torch.Size([32, 1, 32000])
Target shape: torch.Size([32, 2, 32000])


In [93]:
output_signal = inputs[0,0,:]
# output_signal = targets[0,0,:]
# output_signal = targets[0,1,:]

In [94]:
# Play the audio
Audio(output_signal, rate=SAMPLING_RATE)

In [12]:
test_input.shape

torch.Size([32, 1, 32000])

## 🎧 Audio Source Separation using Masks

---

### 🎯 1️⃣ The Goal of Source Separation

Given a **mixture signal**:

$$
x(t) = \sum_{i=1}^{C} s_i(t)
$$

where $C$ is the number of sources (e.g., two speakers), the goal is to recover each individual source $s_i(t)$.

---

### 🧠 2️⃣ The Masking Principle

Instead of directly estimating $s_i(t)$, we estimate **masks** that select which parts of the mixture belong to each source.

A **mask** acts as a filter that emphasizes time–frequency regions belonging to one source and suppresses others.

---

### ⚙️ 3️⃣ In the STFT Domain

#### Step 1: Compute STFT

$$
X(f, t) = \text{STFT}\{x(t)\}
$$

#### Step 2: Estimate Masks

A separation network predicts a **mask** for each source:

$$
M_i(f, t) \in [0, 1]
$$

#### Step 3: Apply the Mask

$$
\hat{S}_i(f, t) = M_i(f, t) \cdot X(f, t)
$$

#### Step 4: Inverse STFT

$$
\hat{s}_i(t) = \text{ISTFT}\{\hat{S}_i(f, t)\}
$$

The Mask Acts As: A soft selection matrix over time–frequency bins: $M_i(f,t) \approx \text{how much of bin } (f,t) \text{ belongs to source } i.$

---
### 4️⃣ In Time-Domain Models (Conv-TasNet)

#### Step 1: Encode Mixture

$$
W = \text{Encoder}(x)
$$

where $W \in \mathbb{R}^{N \times T}$ is the latent representation.

#### Step 2: Estimate Masks

$$
M_i = f_\theta(W), \quad M_i \in [0, 1]^{N \times T}
$$

#### Step 3: Apply Masks

$$
D_i = M_i \odot W
$$

where $\odot$ denotes elementwise multiplication

#### Step 4: Decode Each Source

$$
\hat{s}_i(t) = \text{Decoder}(D_i)
$$

✅ This is a **time-domain** analog of spectrogram masking.

## Conv-Tas Net

- **Encoder**: 1-D convolution mapping: waveform → high-dimensional representation
- **Separator**: (TCN) Stacked dilated conv blocks with exponentially increasing dilation
- **Decoder**: 1-D transposed convolution reconstructing waveforms
- **Normalization**: Global Layer Norm (noncausal) or Cumulative Layer Norm (causal)
- **Activation**: PReLU throughout, Relu and Sigmoid for masks

## 1. Encoder (1-D convolution): audio -> latent representation

<div align="center">
    <img src="Figures/stft_process.png" width="800">
    <p><b>Figure 1:</b> Encode 1-D audio to 2-D latent representation by STFT. </p>
</div>

### The DFT itself is perfectly invertible

For a discrete signal $x[n]$ of length $N$,  
the **DFT** and **Inverse DFT (IDFT)** are given by:

$$
X[k] = \sum_{n=0}^{N-1} x[n] \, e^{-j \frac{2\pi}{N}kn},
$$

$$
x[n] = \frac{1}{N} \sum_{k=0}^{N-1} X[k] \, e^{j \frac{2\pi}{N}kn}.
$$

✅ **If we use the full-length DFT (no truncation, no windowing)**,  
and we keep both magnitude and phase,  
then the DFT–IDFT pair is *mathematically exact* —  
you get **perfect reconstruction (PR)** with no loss.

In [15]:
in_channels_encoder = 1
out_channels_encoder = 256
kernel_size_encoder = 256
stride_encoder = kernel_size_encoder// 4 # 75% overlap
stride_encoder

64

In [16]:
class Encoder(nn.Module):
    """
    The Encoder transforms audio waveforms into a higher-dimensional representation.
    Think of it like breaking down the audio into many different "features".
    """
    def __init__(self, num_filters, kernel_size, stride):
        super(Encoder, self).__init__()
        
        # 1D Convolution: transforms raw audio into features
        self.conv = nn.Conv1d(
            in_channels=1,           # Input: single audio channel
            out_channels=num_filters, # Output: many filter channels
            kernel_size=kernel_size,  # Size of the filter window
            stride=stride,            # How much to move the filter
            bias=False
        )
        self.relu = nn.PReLU()  # Activation function (Should use relu to keep non-negativity but it compromises the extracted information)
        
    def forward(self, x):
        """
        x shape: (batch_size, 1, audio_length)
        output shape: (batch_size, num_filters, num_frames)
        """
        x = self.conv(x)
        x = self.relu(x)
        return x

In [17]:
f_encoder = Encoder(num_filters=out_channels_encoder, kernel_size=kernel_size_encoder, stride=stride_encoder)
f_encoder

Encoder(
  (conv): Conv1d(1, 256, kernel_size=(256,), stride=(64,), bias=False)
  (relu): PReLU(num_parameters=1)
)

In [18]:
output_encoder = f_encoder(test_input)

In [19]:
output_encoder.shape

torch.Size([32, 256, 497])

- Input: $(N, C_{\text{in}}, L_{\text{in}}) \quad \text{or} \quad (C_{\text{in}}, L_{\text{in}})$

- Output: $(N, C_{\text{out}}, L_{\text{out}}) \quad \text{or} \quad (C_{\text{out}}, L_{\text{out}})$

$N$ is a batch size, $C$ denotes a number of channels, $L$ is a length of signal sequence.
 
$$L_{\text{out}} = \left\lfloor \frac{L_{\text{in}} + 2 \cdot \text{padding} - \text{dilation} \cdot (\text{kernel\_size} - 1) - 1}{\text{stride}} + 1 \right\rfloor$$

In [20]:
TIME_SIZE = int((WINDOW_SIZE - kernel_size_encoder - 2) / stride_encoder + 1 + 0.5)
TIME_SIZE

497

In [21]:
# Get all parameters
print("=== All Parameters ===")
for name, param in f_encoder.named_parameters():
    print(f"{name}: {param.shape}")

=== All Parameters ===
conv.weight: torch.Size([256, 1, 256])
relu.weight: torch.Size([1])


In [22]:
f_encoder.conv.weight.data

tensor([[[-0.0305, -0.0371, -0.0028,  ..., -0.0015, -0.0543, -0.0326]],

        [[-0.0567,  0.0411, -0.0491,  ...,  0.0429, -0.0218,  0.0493]],

        [[-0.0450,  0.0471,  0.0403,  ..., -0.0393,  0.0142, -0.0071]],

        ...,

        [[ 0.0249,  0.0465,  0.0048,  ...,  0.0532, -0.0300, -0.0324]],

        [[ 0.0079,  0.0044, -0.0433,  ...,  0.0015, -0.0003,  0.0535]],

        [[ 0.0022, -0.0423, -0.0405,  ...,  0.0094, -0.0453,  0.0041]]])

## 2. Depthwise Separable Convolution

- *Paper: [Xception: Deep Learning with Depthwise Separable Convolutions](https://arxiv.org/pdf/1610.02357)*

```text
Standard Convolution:
┌─────────────────────────────────────────────────────┐
│  Input ──────► [Full Convolution] ──────► Output   │
│  (C_in)           (expensive)            (C_out)   │
└─────────────────────────────────────────────────────┘

Depthwise Separable Convolution:
┌─────────────────────────────────────────────────────┐
│  Input ──► [Depthwise] ──► [Pointwise] ──► Output  │
│  (C_in)    (per channel)    (1x1 conv)    (C_out)  │
│               (cheap)        (cheap)               │
└─────────────────────────────────────────────────────┘
```

### Standard 1D Convolution

For an input $ X \in \mathbb{R}^{C_{\text{in}} \times L} $ and output $Y \in \mathbb{R}^{C_{\text{out}} \times L'}$, a standard 1D convolution is defined as:

$$
Y_{j}(t) = \sum_{c=1}^{C_{\text{in}}} \sum_{k=1}^{K} 
W_{j,c,k} \, X_{c}(t + k)
$$

where $W_{j,c,k}$ are learnable weights of shape $C_{\text{out}} \times C_{\text{in}} \times K$.

---

### Depthwise Separable Convolution

This operation factorizes the standard convolution into:

1. **Depthwise convolution** — per-channel convolution  
2. **Pointwise convolution** — $1 \times 1$ convolution mixing channels

---

#### Depthwise Convolution

$$
Z_{c}(t) = \sum_{k=1}^{K} K_{c,k} \, X_{c}(t + k),
\quad K \in \mathbb{R}^{C_{\text{in}} \times K}
$$

Each input channel is convolved independently, producing 
$Z \in \mathbb{R}^{C_{\text{in}} \times L'}$.

---

#### Pointwise Convolution

$$
Y_{j}(t) = \sum_{c=1}^{C_{\text{in}}} P_{j,c} \, Z_{c}(t),
\quad P \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}}}
$$

The $1 \times 1$ convolution mixes information across channels.

---

#### Combined Operation

Combining both stages gives:

$$
Y_{j}(t) = 
\sum_{c=1}^{C_{\text{in}}} P_{j,c} 
\left( \sum_{k=1}^{K} K_{c,k} \, X_{c}(t + k) \right)
$$

---

### Parameter Comparison

$$
\text{Standard: } C_{\text{out}} \times C_{\text{in}} \times K
\quad \text{vs.} \quad
\text{Depthwise Separable: } (C_{\text{in}} \times K) + (C_{\text{out}} \times C_{\text{in}})
$$


In [23]:
def padding_same(x, dilation, kernel_size):
    """
    Calculate padding to maintain same size
    """
    # Calculate total padding needed
    effective_kernel_size = dilation * (kernel_size - 1) + 1
    total_padding = effective_kernel_size - 1
    
    # Split into left and right padding
    pad_left = total_padding // 2
    pad_right = total_padding - pad_left  # Handles odd padding
    
    # Apply asymmetric padding: F.pad format is (left, right)
    x_padded = F.pad(x, (pad_left, pad_right))
    
    return x_padded

In [24]:
class Depthwise_conv(nn.Module):
    """
    Step 1: Depthwise convolution with dilation
    Each input channel gets its own filter (groups=in_channels)
    """
    def __init__(self, in_channels, kernel_size, dilation, time_size):
        super(Depthwise_conv, self).__init__()
        self.depthwise = nn.Conv1d(
                in_channels=in_channels, 
                out_channels=in_channels, 
                kernel_size=kernel_size,
                padding=0,           # We will pad manually to maintain the same TIME_SIZE
                dilation=dilation,
                groups=in_channels,  # Depthwise means each channel is separate
                bias=False)
        self.norm = nn.LayerNorm((in_channels, time_size))
        self.relu = nn.PReLU()

        self.kernel_size = kernel_size
        self.dilation = dilation

    def forward(self, x):
        # Calculate padding to maintain same size
        x_padded = padding_same(x, self.dilation, self.kernel_size)

        x = self.depthwise(x_padded)
        x = self.norm(x)
        x = self.relu(x)
        
        return x

In [25]:
in_channels_l1 = out_channels_encoder
kernel_size_l1 = 40
dilation_l1 = 1

In [27]:
f_depthwise = Depthwise_conv(in_channels_l1, kernel_size_l1, dilation_l1, TIME_SIZE)
f_depthwise

Depthwise_conv(
  (depthwise): Conv1d(256, 256, kernel_size=(40,), stride=(1,), groups=256, bias=False)
  (norm): LayerNorm((256, 497), eps=1e-05, elementwise_affine=True)
  (relu): PReLU(num_parameters=1)
)

In [28]:
output_depthwise = f_depthwise(output_encoder)
output_depthwise.shape

torch.Size([32, 256, 497])

In [29]:
# Get all parameters
print("=== All Parameters ===")
for name, param in f_depthwise.named_parameters():    
    print(f"{name}: {param.shape}")

=== All Parameters ===
depthwise.weight: torch.Size([256, 1, 40])
norm.weight: torch.Size([256, 497])
norm.bias: torch.Size([256, 497])
relu.weight: torch.Size([1])


In [30]:
class Pointwise_conv(nn.Module):
    """
    Step 2: Pointwise Convolution (1x1 convolution)
    Combines information across channels
    """
    def __init__(self, in_channels, out_channels, time_size):
        super(Pointwise_conv, self).__init__()
        
        self.pointwise = nn.Conv1d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=1,               # 1x1 convolution
                bias=False
            )
        self.norm = nn.LayerNorm((out_channels, time_size))
        self.relu = nn.PReLU()

    def forward(self, x):

        x = self.pointwise(x)
        x = self.norm(x)
        x = self.relu(x)
        
        return x

In [31]:
# Our goal is to increase the number of channels for the latent representation to a multiple of number of sources.
# e.g. out_channels_encoder = 256, and we have 2 sources. The number of output channels after TCN blocks should be 256 x 2 = 512.
num_TCN = 4
channel_expansion = out_channels_encoder // num_TCN
channel_expansion

64

In [32]:
out_channels_l1 = in_channels_l1 + channel_expansion
out_channels_l1

320

In [34]:
f_pointwise = Pointwise_conv(in_channels_l1, out_channels_l1, TIME_SIZE)
f_pointwise

Pointwise_conv(
  (pointwise): Conv1d(256, 320, kernel_size=(1,), stride=(1,), bias=False)
  (norm): LayerNorm((320, 497), eps=1e-05, elementwise_affine=True)
  (relu): PReLU(num_parameters=1)
)

In [35]:
output_pointwise = f_pointwise(output_depthwise)
output_pointwise.shape

torch.Size([32, 320, 497])

In [36]:
# Get all parameters
print("=== All Parameters ===")
for name, param in f_pointwise.named_parameters():    
    print(f"{name}: {param.shape}")

=== All Parameters ===
pointwise.weight: torch.Size([320, 256, 1])
norm.weight: torch.Size([320, 497])
norm.bias: torch.Size([320, 497])
relu.weight: torch.Size([1])


In [37]:
class TCNBlock(nn.Module):
    """
    A single block of the Temporal Convolutional Network.
    It processes the features over time with dilated convolutions.
    """
    def __init__(self, in_channels, out_channels, kernel_size, dilation, time_size):
        super(TCNBlock, self).__init__()

        # Step 1: Depthwise convolution with dilation
        # Each input channel gets its own filter (groups=in_channels)
        self.depthwise = Depthwise_conv(in_channels, kernel_size, dilation, time_size)
        
        # Step 2: Pointwise Convolution (1x1 convolution)
        # Combines information across channels
        self.pointwise = Pointwise_conv(in_channels, out_channels, time_size)

        # residual
        self.residual = Pointwise_conv(in_channels, out_channels, time_size)
        
    def forward(self, x):
        """
        Process the input through the TCN block with residual connection.
        """
        residual = self.residual(x)  # Save input for skip connection
        
        # Step 1: Depthwise
        x = self.depthwise(x)
        
        # Step 2: Pointwise
        x = self.pointwise(x)
        
        # Add residual (skip connection) - this helps learning!
        x = x + residual
        
        return x

In [ ]:
# class TCNBlock(nn.Module):
#     """
#     A single block of the Temporal Convolutional Network.
#     Includes both residual and skip connections as per Conv-TasNet.
#     """
#     def __init__(self, in_channels, out_channels, skip_channels, kernel_size, dilation, time_size):
#         super(TCNBlock, self).__init__()

#         # Step 1: Depthwise convolution with dilation
#         self.depthwise = Depthwise_conv(in_channels, kernel_size, dilation, time_size)
        
#         # Step 2: Pointwise Convolution (1x1 convolution)
#         self.pointwise = Pointwise_conv(in_channels, out_channels, time_size)

#         # Residual connection (1x1 conv to match dimensions)
#         self.residual = Pointwise_conv(in_channels, out_channels, time_size)
        
#         # Skip connection (1x1 conv for skip output)
#         self.skip = Pointwise_conv(in_channels, skip_channels, time_size)
        
#     def forward(self, x):
#         """
#         Process input through TCN block with residual and skip connections.
        
#         Args:
#             x: Input tensor of shape (batch, channels, time)
        
#         Returns:
#             output: Residual output for next block
#             skip: Skip connection output to be summed later
#         """
#         # Residual path
#         residual = self.residual(x)
        
#         # Main path
#         out = self.depthwise(x)
#         out = self.pointwise(out)
        
#         # Skip connection output (before adding residual)
#         skip = self.skip(out)
        
#         # Add residual
#         output = out + residual
        
#         return output, skip


In [38]:
f_TCN = TCNBlock(in_channels_l1, out_channels_l1, kernel_size_l1, dilation_l1, TIME_SIZE)
f_TCN

TCNBlock(
  (depthwise): Depthwise_conv(
    (depthwise): Conv1d(256, 256, kernel_size=(40,), stride=(1,), groups=256, bias=False)
    (norm): LayerNorm((256, 497), eps=1e-05, elementwise_affine=True)
    (relu): PReLU(num_parameters=1)
  )
  (pointwise): Pointwise_conv(
    (pointwise): Conv1d(256, 320, kernel_size=(1,), stride=(1,), bias=False)
    (norm): LayerNorm((320, 497), eps=1e-05, elementwise_affine=True)
    (relu): PReLU(num_parameters=1)
  )
  (residual): Pointwise_conv(
    (pointwise): Conv1d(256, 320, kernel_size=(1,), stride=(1,), bias=False)
    (norm): LayerNorm((320, 497), eps=1e-05, elementwise_affine=True)
    (relu): PReLU(num_parameters=1)
  )
)

In [39]:
output_l1 = f_TCN(output_encoder)
output_l1.shape

torch.Size([32, 320, 497])

## 3. Dilated Temporal Convolutional Network (Dilated TCN)

---

*Paper: [WaveNet: A Generative Model for Raw Audio](https://arxiv.org/pdf/1609.03499)*
<div align="center">
    <img src="Figures/wavenet.png" width="800">
    <p><b>Figure 1:</b> Dilated Causal Convolutional Layers. </p>
</div>

---

### 🧠 Overall TCN Structure

In **Conv-TasNet**, the **masking network** consists of:
- $M$ **convolutional blocks** per *repeat*
- Each block uses a **dilation factor** $d_m = 2^{m-1}$ for $m = 1, 2, \dots, M$
- The entire sequence of $M$ blocks is **repeated** $R$ times

Hence, the **total number of convolutional layers** is $R \times M$.

---

### ⚙️ Input and Output Relationship of a Dilated 1-D Convolution

Consider a **1-D convolution** with:
- Kernel size: $K$
- Dilation: $d$
- Padding: $p$
- Stride: 1

Then for an input $x[n]$, the convolution output $y[n]$ is defined as:

$$
y[n] = \sum_{k=0}^{K-1} w[k] \, x[n - d \cdot k]
$$

where:
- $w[k]$ are the convolution filter coefficients,
- $d$ determines the **spacing** between filter elements in time.

the **receptive field** is:

$$
R = (K - 1)d + 1
$$

Thus, each layer increases the temporal coverage by a factor proportional to $d$.

---

## 🧠 Zero Padding for Equal-Length Output

For a convolution with stride $1$ and dilation $d$,  
to preserve **input and output lengths** ($L_{\text{out}} = L_{\text{in}}$),  
the padding $p$ must satisfy:

$$
L_{\text{out}} = L_{\text{in}} + 2p - d(K - 1)
$$

Setting $L_{\text{out}} = L_{\text{in}}$ gives:

$$
p = \frac{d(K - 1)}{2}
$$

This ensures that the **output sequence** has the same temporal length as the **input**, regardless of dilation.

✅ In Conv-TasNet, this “same” padding keeps all residual connections shape-consistent.

---

<div align="center">
    <img src="Figures/dilation_pad.png" width="800">
    <p><b>Figure 2:</b> Dilated Temporal Convolutional Network Zero Padded for Equal-Length Output. </p>
</div>

## 3. Apply Masks

$$
D_i = M_i \odot W
$$

where $W$ is the latent representation of the mixed signal from the encoder, and $\odot$ denotes elementwise multiplication.

In [3]:
class Mask_conv(nn.Module):
    """
    Step 3: Mask Convolution (1x1 convolution)
    produces masks for each source
    Align the number of output channels with num_sources * num_filters
    """
    def __init__(self, in_channels, out_channels, time_size):
        super(Mask_conv, self).__init__()
        
        self.pointwise = nn.Conv1d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=1,               # 1x1 convolution
                bias=False
            )
        self.norm = nn.LayerNorm((out_channels, time_size))
        self.relu = nn.ReLU() # non-negative mask
        self.sigmoid = nn.Sigmoid() # mask values [0,1]

    def forward(self, x):
        
        x = self.pointwise(x)
        x = self.norm(x)
        x = self.relu(x)
        x = self.sigmoid(x)
        
        return x

In [4]:
class Separator(nn.Module):
    """
    The Separator processes the encoded features to separate different sources.
    It uses multiple TCN blocks stacked together.
    """
    def __init__(self, num_sources, num_blocks, num_repeats, in_channels, kernel_size, time_size):
        super(Separator, self).__init__()

        self.num_sources = num_sources
        
        # Our goal is to increase the number of channels for the latent representation to a multiple of number of sources.
        # e.g. out_channels_encoder = 256, and we have 2 sources. The number of output channels after TCN blocks should be 256 x 2 = 512.
        out_channels = in_channels * num_sources 
        channel_expansion = (out_channels - in_channels) // num_blocks
        
        # Build the TCN blocks
        self.tcn_blocks = nn.ModuleList()
        for repeat in range(num_repeats):
            for block in range(num_blocks):
                dilation = 2 ** block  # Exponentially increasing dilation
                self.tcn_blocks.append(
                    TCNBlock(in_channels + channel_expansion*block, in_channels + channel_expansion*(block+1), kernel_size, dilation, time_size)
                )
        
        # Output layer: produces masks for each source
        self.mask_conv = Mask_conv(in_channels + channel_expansion*num_blocks, out_channels, time_size)
        
    def forward(self, x):
        """
        x shape: (batch_size, num_filters, TIME_SIZE)
        output shape: (batch_size, num_sources, num_filters, num_frames)
        """
        batch_size, num_filters, num_frames = x.shape
        
        # Pass through all TCN blocks
        for block in self.tcn_blocks:
            x = block(x)
        
        # Generate masks for each source
        masks = self.mask_conv(x)  # (batch, num_filters * num_sources, frames)
        
        # Reshape to separate sources
        masks = masks.view(batch_size, self.num_sources, num_filters, num_frames)
        
        return masks

In [41]:
# Our goal is to increase the number of channels for the latent representation to a multiple of number of sources.
# e.g. out_channels_encoder = 256, and we have 2 sources. The number of output channels after TCN blocks should be 256 x 2 = 512.
num_sources = 2  # 2 speakers
num_repeats = 1
num_blocks = 4
channel_expansion = out_channels_encoder // num_blocks
channel_expansion

64

In [43]:
f_separator = Separator(num_sources, num_blocks, num_repeats, in_channels_l1, kernel_size_l1, TIME_SIZE)
f_separator

Separator(
  (tcn_blocks): ModuleList(
    (0): TCNBlock(
      (depthwise): Depthwise_conv(
        (depthwise): Conv1d(256, 256, kernel_size=(40,), stride=(1,), groups=256, bias=False)
        (norm): LayerNorm((256, 497), eps=1e-05, elementwise_affine=True)
        (relu): PReLU(num_parameters=1)
      )
      (pointwise): Pointwise_conv(
        (pointwise): Conv1d(256, 320, kernel_size=(1,), stride=(1,), bias=False)
        (norm): LayerNorm((320, 497), eps=1e-05, elementwise_affine=True)
        (relu): PReLU(num_parameters=1)
      )
      (residual): Pointwise_conv(
        (pointwise): Conv1d(256, 320, kernel_size=(1,), stride=(1,), bias=False)
        (norm): LayerNorm((320, 497), eps=1e-05, elementwise_affine=True)
        (relu): PReLU(num_parameters=1)
      )
    )
    (1): TCNBlock(
      (depthwise): Depthwise_conv(
        (depthwise): Conv1d(320, 320, kernel_size=(40,), stride=(1,), dilation=(2,), groups=320, bias=False)
        (norm): LayerNorm((320, 497), eps=1e-05

In [44]:
mask_representation = f_separator(output_encoder)
mask_representation.shape

torch.Size([32, 2, 256, 497])

In [45]:
mask_representation.min()

tensor(0.5000, grad_fn=<MinBackward1>)

## 4. Decoder (1-D Transpose convolution): latent representation -> audio

The decoder reconstructs the waveform from this representation using a 1-D transposed convolution operation, which can be reformulated as another matrix multiplication:

$$
\hat{\mathbf{x}} = \mathbf{w}\mathbf{V}
$$

where:
- $\hat{\mathbf{x}} \in \mathbb{R}^{1 \times L}$ is the reconstruction of $\mathbf{x}$
- $\mathbf{V} \in \mathbb{R}^{N \times L}$ contains the decoder basis functions (rows), each with length $L$

The overlapping reconstructed segments are summed together to generate the final waveforms.


In [46]:
class Decoder(nn.Module):
    """
    The Decoder transforms the separated features back into audio waveforms.
    It's the reverse of the Encoder.
    """
    def __init__(self, num_filters, kernel_size, stride):
        super(Decoder, self).__init__()
        
        # Transposed Convolution: transforms features back to audio
        self.deconv = nn.ConvTranspose1d(
            in_channels=num_filters,
            out_channels=1,
            kernel_size=kernel_size,
            stride=stride,
            bias=False
        )
        
    def forward(self, x):
        """
        x shape: (batch_size, num_filters, num_frames)
        output shape: (batch_size, 1, audio_length)
        """
        x = self.deconv(x)
        return x

In [47]:
f_decoder = Decoder(num_filters=out_channels_encoder, kernel_size=kernel_size_encoder, stride=stride_encoder)
f_decoder

Decoder(
  (deconv): ConvTranspose1d(256, 1, kernel_size=(256,), stride=(64,), bias=False)
)

In [48]:
output_encoder.shape

torch.Size([32, 256, 497])

In [49]:
mask_representation.shape

torch.Size([32, 2, 256, 497])

In [50]:
# Apply masks and decode each source
separated_sources = []
for source_idx in range(num_sources):
    # Apply mask to encoded representation
    masked = output_encoder * mask_representation[:, source_idx, :, :]
    # Decode back to audio
    output_decoder = f_decoder(masked)
    separated_sources.append(output_decoder)

In [51]:
output_decoder.shape

torch.Size([32, 1, 32000])

## 5. Build the Complete Conv-Tas Model

In [69]:
class Config:
    """All the settings for our model in one place"""

    # Audio settings
    DEVICE = torch.device('cuda')
    SAMPLING_RATE = SAMPLING_RATE
    WINDOW_SIZE = WINDOW_SIZE
    HOP_SIZE = HOP_SIZE
    
    # Encoder settings
    NUM_FILTERS = 256          # Number of filters in encoder
    KERNEL_SIZE_ENCODER = 256  # Analogous to STFT for perfect reconstruction
    STRIDE = KERNEL_SIZE_ENCODER // 4   # 75% overlap
    TIME_SIZE = int((WINDOW_SIZE - KERNEL_SIZE_ENCODER - 2) / STRIDE + 1 + 0.5)
    
    # Separator settings
    NUM_SOURCES = 2            # Example: 2 speakers
    NUM_BLOCKS = 4             # Number of TCN blocks
    NUM_REPEATS = 1            # How many times to repeat blocks
    KERNEL_SIZE_TCN = 40
    # FILTER_INCREMENT = NUM_FILTERS * (NUM_SOURCES - 1) // NUM_BLOCKS 
    # After NUM_BLOCKS blocks, the number of filters is expanded to "NUM_FILTERS x NUM_SOURCES"
    
    # Training settings
    BATCH_SIZE = 32             # How many samples to process at once
    LEARNING_RATE = 0.001      # How fast the model learns
    NUM_EPOCHS = 50            # How many times to go through all data

config = Config()

In [53]:
print('TIME_SIZE: ', config.TIME_SIZE)
print('SAMPLING_RATE: ', config.SAMPLING_RATE)

TIME_SIZE:  497
SAMPLING_RATE:  8000


In [54]:
class ConvTasNet(nn.Module):
    """
    The complete Conv-TasNet model combining Encoder, Separator, and Decoder.
    """
    def __init__(self, config):
        super(ConvTasNet, self).__init__()
        
        self.num_sources = config.NUM_SOURCES
        
        # Create the three main components
        self.encoder = Encoder(
            config.NUM_FILTERS, 
            config.KERNEL_SIZE_ENCODER, 
            config.STRIDE
        )
        
        self.separator = Separator(
            config.NUM_SOURCES,
            config.NUM_BLOCKS,
            config.NUM_REPEATS,
            config.NUM_FILTERS,
            config.KERNEL_SIZE_TCN,
            config.TIME_SIZE
        )
        
        self.decoder = Decoder(
            config.NUM_FILTERS,
            config.KERNEL_SIZE_ENCODER,
            config.STRIDE
        )
        
    def forward(self, mixed_audio):
        """
        Separate mixed audio into individual sources.
        
        Input: mixed_audio of shape (batch_size, 1, audio_length)
        Output: separated sources of shape (batch_size, num_sources, audio_length)
        """
        # Step 1: Encode the mixed audio
        encoded = self.encoder(mixed_audio)
        # Shape: (batch_size, num_filters, num_frames)
        
        # Step 2: Generate separation masks
        masks = self.separator(encoded)
        # Shape: (batch_size, num_sources, num_filters, num_frames)
        
        # Step 3: Apply masks and decode each source
        separated_sources = []
        for source_idx in range(self.num_sources):
            # Apply mask to encoded representation
            masked = encoded * masks[:, source_idx, :, :]
            # Decode back to audio
            decoded = self.decoder(masked)
            separated_sources.append(decoded)
        
        # Stack all sources together
        output = torch.stack(separated_sources, dim=1)
        # Shape: (batch_size, num_sources, 1, audio_length)
        
        # Remove the extra dimension
        output = output.squeeze(2)
        # Shape: (batch_size, num_sources, audio_length)
        
        return output

In [55]:
model = ConvTasNet(config)

In [56]:
model

ConvTasNet(
  (encoder): Encoder(
    (conv): Conv1d(1, 256, kernel_size=(256,), stride=(64,), bias=False)
    (relu): PReLU(num_parameters=1)
  )
  (separator): Separator(
    (tcn_blocks): ModuleList(
      (0): TCNBlock(
        (depthwise): Depthwise_conv(
          (depthwise): Conv1d(256, 256, kernel_size=(40,), stride=(1,), groups=256, bias=False)
          (norm): LayerNorm((256, 497), eps=1e-05, elementwise_affine=True)
          (relu): PReLU(num_parameters=1)
        )
        (pointwise): Pointwise_conv(
          (pointwise): Conv1d(256, 320, kernel_size=(1,), stride=(1,), bias=False)
          (norm): LayerNorm((320, 497), eps=1e-05, elementwise_affine=True)
          (relu): PReLU(num_parameters=1)
        )
        (residual): Pointwise_conv(
          (pointwise): Conv1d(256, 320, kernel_size=(1,), stride=(1,), bias=False)
          (norm): LayerNorm((320, 497), eps=1e-05, elementwise_affine=True)
          (relu): PReLU(num_parameters=1)
        )
      )
      (1): T

In [57]:
output = model(test_input)
output.shape

torch.Size([32, 2, 32000])

## 6. Scale-Invariant Signal-to-Noise Ratio (SI-SNR) Loss in Audio Source Separation

- *Paper: [A Study of the Scale Invariant Signal to Distortion Ratio in Speech Separation with Noisy References](https://arxiv.org/pdf/2508.14623)*

---

### 🧠 1️⃣ The Problem: Why SI-SNR?

In source separation, we have a mixture signal:

$$
x(t) = s(t) + n(t)
$$

- $x(t)$ : mixture waveform  
- $s(t)$ : target (clean) source  
- $n(t)$ : interference / noise  

Our model outputs an estimate $\hat{s}(t)$,  
and we want to measure how close $\hat{s}(t)$ is to $s(t)$.

---

### ⚙️ 2️⃣ Classic SNR (not scale-invariant)

The traditional Signal-to-Noise Ratio (SNR) is defined as:

$$
\text{SNR} = 10 \log_{10} \frac{\|s\|^2}{\|s - \hat{s}\|^2}
$$

⚠️ **Problem:**  
If the model predicts a scaled version of the signal (e.g. $2s$),  
the SNR drops even though the waveform *shape* is perfect.

Therefore, SNR is **not scale-invariant**.

---

### 🧩 3️⃣ Derivation of SI-SNR

Let $s$ be the true clean source and $\hat{s}$ be the estimated signal.

First, remove the mean (DC offset):

$$
s_z = s - \text{mean}(s)
$$

$$
\hat{s}_z = \hat{s} - \text{mean}(\hat{s})
$$

---

#### Step 1️⃣ — Project $\hat{s}_z$ onto $s_z$

We decompose $\hat{s}_z$ into:

- **target component** $s_{\text{target}}$: projection of $\hat{s}_z$ onto $s_z$
- **noise component** $e_{\text{noise}}$: orthogonal residual

$$
s_{\text{target}} = \frac{\langle \hat{s}_z, s_z \rangle}{\|s_z\|^2} \, s_z
$$

$$
e_{\text{noise}} = \hat{s}_z - s_{\text{target}}
$$

The projection step ensures **scale invariance** because it finds the optimal scaling factor automatically.

---

#### Step 2️⃣ — Compute SI-SNR

$$
\text{SI-SNR} = 10 \log_{10} \frac{\| s_{\text{target}} \|^2}{\| e_{\text{noise}} \|^2}
$$

✅ A higher SI-SNR indicates a cleaner separation (less residual noise).

---

#### Step 3️⃣ — Define as a Loss Function

Since we want to *maximize* SI-SNR,  
the **loss** is simply its negative:

$$
\mathcal{L}_{\text{SI-SNR}} = -\text{SI-SNR}
$$

---

### Step 4️⃣ Intuitive Meaning

- Numerator $\| s_{\text{target}} \|^2$ measures how much of the true signal is captured.  
- Denominator $\| e_{\text{noise}} \|^2$ measures how much residual error remains.  
- By projecting onto $s_z$, we ignore absolute scaling — only the **waveform shape** matters.

Thus, SI-SNR measures **shape similarity**, not amplitude similarity.




### 🔍 Difference between SNR and SI-SNR

| Metric | Sensitive to scale? | Purpose |
|---------|--------------------|----------|
| **SNR** | ✅ Yes | Measures ratio of absolute signal vs. noise powers |
| **SI-SNR** | ❌ No (scale-invariant) | Measures *shape* similarity only (ignores amplitude) |

So:
- SNR penalizes scaling differences (e.g. $\hat{s} = 2s$).
- SI-SNR corrects this by projecting $\hat{s}$ onto $s$ first.

## Classic SNR Loss
$$
\mathcal{L}_{\text{SNR}} = 10 \log_{10} \frac{\|s - \hat{s}\|^2}{\|s\|^2}
$$

In [58]:
def snr_loss(estimated, target):
    """
    Classic Signal-to-Noise Ratio loss.
    Higher SNR = better separation (we minimize negative SNR)
    """
    # Ensure same length
    min_len = min(estimated.shape[-1], target.shape[-1])
    estimated = estimated[..., :min_len]
    target = target[..., :min_len]
    
    # noise = s' - s_target
    noise = target - estimated
    
    # SNR = 10 * log10(||noise||^2 / ||s_target||^2)
    loss = 10 * torch.log10(
        ((noise ** 2).sum(dim=-1) / (target ** 2).sum(dim=-1)) + 1e-8
    )
    
    return loss.mean()

In [59]:
test_target.shape

torch.Size([32, 2, 32000])

In [60]:
loss_snr = snr_loss(output, test_target)
loss_snr

tensor(0.8493, grad_fn=<MeanBackward0>)

In [61]:
def si_snr_loss(estimated, target):
    """
    Scale-Invariant Signal-to-Noise Ratio loss.
    This is the recommended loss function for audio separation.
    
    Higher SI-SNR = better separation (we minimize negative SI-SNR)
    """
    # Ensure same length
    min_len = min(estimated.shape[-1], target.shape[-1])
    estimated = estimated[..., :min_len]
    target = target[..., :min_len]
    
    # Zero-mean normalization
    estimated = estimated - estimated.mean(dim=-1, keepdim=True)
    target = target - target.mean(dim=-1, keepdim=True)
    
    # Calculate SI-SNR
    # s_target = <s', s> * s / ||s||^2
    dot_product = (estimated * target).sum(dim=-1, keepdim=True)
    target_energy = (target ** 2).sum(dim=-1, keepdim=True) + 1e-8
    projection = dot_product * target / target_energy
    
    # noise = s' - s_target
    noise = estimated - projection
    
    # SI-SNR = 10 * log10(||s_target||^2 / ||noise||^2)
    si_snr = 10 * torch.log10(
        (projection ** 2).sum(dim=-1) / ((noise ** 2).sum(dim=-1) + 1e-8) + 1e-8
    )
    
    # Return negative mean (we want to maximize SI-SNR, so minimize negative)
    return -si_snr.mean()

In [62]:
loss_si_snr = si_snr_loss(output, test_target)
loss_si_snr

tensor(42.8305, grad_fn=<NegBackward0>)

## 7. Training Function

## Neural Network Optimizers

*Paper: [Adam: A Method for Stochastic Optimization](https://arxiv.org/pdf/1412.6980)*

---

### 🎯 1️⃣ Problem Setup

In neural network training, we minimize a **loss function** $L(\theta)$  with respect to the model parameters $\theta \in \mathbb{R}^d$.

The general optimization problem is:

$$
\min_{\theta} L(\theta)
$$

We update parameters iteratively using gradients of the loss:

$$
g_t = \nabla_{\theta_t} L(\theta_t)
$$

---

### ⚙️ 2️⃣ Basic Gradient Descent (GD)

#### Update rule:
$$
\theta_{t+1} = \theta_t - \eta \, g_t
$$

where:
- $\eta$ = learning rate  
- $g_t = \nabla_{\theta_t} L(\theta_t)$ = gradient at iteration $t$

#### Intuition:
Move parameters in the **opposite direction** of the gradient to reduce the loss.

---

### ⚡ 3️⃣ Stochastic Gradient Descent (SGD)

Since computing the full gradient is expensive, we use a **mini-batch** $\mathcal{B}_t$:

$$
g_t = \frac{1}{|\mathcal{B}_t|} \sum_{(x_i, y_i) \in \mathcal{B}_t} \nabla_{\theta_t} L(x_i, y_i; \theta_t)
$$

Then:

$$
\theta_{t+1} = \theta_t - \eta \, g_t
$$

✅ **SGD** introduces randomness → faster convergence, but noisier updates.

---

### 💨 4️⃣ SGD with Momentum

Momentum adds a **velocity term** that accumulates past gradients to smooth updates.

Let:
- $v_t$ = velocity (running average of past gradients)
- $\beta$ = momentum coefficient (e.g. 0.9)

#### Equations:

$$
v_t = \beta v_{t-1} + (1 - \beta) g_t
$$
$$
\theta_{t+1} = \theta_t - \eta v_t
$$

✅ Momentum helps accelerate in consistent directions and dampens oscillations.

---

### ⚙️ 5️⃣ Nesterov Accelerated Gradient (NAG)

Nesterov momentum computes the gradient *after looking ahead* by one momentum step:

$$
v_t = \beta v_{t-1} + (1 - \beta) \nabla_{\theta_t - \eta \beta v_{t-1}} L(\theta_t)
$$
$$
\theta_{t+1} = \theta_t - \eta v_t
$$

✅ Helps correct oversteps and provides smoother convergence.

---

### ⚡ 6️⃣ Adagrad

Adagrad adapts the learning rate **per parameter**, scaling inversely with the square root of past gradients.

Let:
- $g_{t,i}$ = gradient of parameter $\theta_i$ at step $t$
- $G_t$ = sum of squared gradients

#### Equations:

$$
G_t = G_{t-1} + g_t \odot g_t
$$

$$
\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t + \epsilon}} \odot g_t
$$

✅ Each parameter gets its own adaptive learning rate.  
⚠️ But $G_t$ grows unbounded → learning rate keeps shrinking over time.

---

### ⚙️ 7️⃣ RMSProp

RMSProp fixes Adagrad’s decaying learning rate by using **exponential moving average** of squared gradients.

Let $\beta$ ≈ 0.9.

$$
E[g^2]_t = \beta E[g^2]_{t-1} + (1 - \beta) g_t^2
$$

$$
\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{E[g^2]_t + \epsilon}} \odot g_t
$$

✅ Keeps a **windowed average** of squared gradients → stable adaptive steps.

---


### 🧮 8️⃣ Adam (Adaptive Moment Estimation)

Adam combines **Momentum** + **RMSProp**:  
it tracks both *first moment* (mean) and *second moment* (variance) of gradients.

Parameters:
- $\beta_1$ ≈ 0.9 → momentum coefficient  
- $\beta_2$ ≈ 0.999 → RMS smoothing coefficient  

---

#### Step 1: Compute biased moments

$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t \\
v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2
$$

#### Step 2: Bias correction

$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad 
\hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$


#### Step 3: Parameter update

$$
\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

✅ Adam = adaptive + momentum → fast, stable, and widely used.  
⚠️ Sometimes overfits or converges to suboptimal minima (especially in high noise).

---

### ⚙️ 9️⃣ AdamW (Adam with Weight Decay)

In standard Adam, **L2 regularization** is incorrectly implemented (it interacts with momentum).  
**AdamW** fixes this by decoupling weight decay:

$$
\theta_{t+1} = \theta_t - \eta \left( \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_t \right)
$$

where $\lambda$ is the weight decay coefficient.

✅ This ensures proper regularization behavior and better generalization.

---

✅ **In short:**

> Neural network optimizers are all based on the same principle:
>  
> $$
\theta_{t+1} = \theta_t - \eta \, f(\nabla L, \text{past gradients})
$$
>
> where $f(\cdot)$ differs by optimizer:
> - SGD: raw gradient  
> - Momentum: adds velocity  
> - Adagrad/RMSProp: scales by gradient history  
> - Adam: combines both momentum + adaptive scaling  
> - AdamW: adds proper weight decay for generalization


### What Each Phase Does

```text
┌─────────────────────────────────────────────────────────────┐
│  TRAINING PHASE                                             │
│  • model.train() → enables dropout, batch norm training     │
│  • Computes gradients                                       │
│  • Updates weights                                          │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  VALIDATION PHASE                                           │
│  • model.eval() → disables dropout, uses running stats      │
│  • torch.no_grad() → saves memory, faster                   │
│  • NO weight updates                                        │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│  EARLY STOPPING CHECK                                       │
│  • If val_loss improved → save model, reset patience        │
│  • If no improvement → increment patience counter           │
│  • If patience exceeded → stop training                     │
└─────────────────────────────────────────────────────────────┘
```

In [63]:
def train_model(model, train_loader, config):
    """
    Train the Conv-TasNet model.
    """
    # Move model to GPU if available
    model = model.to(device)
    
    # Set up optimizer
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    
    # Learning rate scheduler (reduces learning rate over time)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )
    
    # Track losses for plotting
    loss_history = []
    
    print("\n" + "=" * 60)
    print("Starting Training")
    print("=" * 60)
    
    for epoch in range(config.NUM_EPOCHS):
        model.train()  # Set model to training mode
        epoch_loss = 0.0
        num_batches = 0
        
        for batch_idx, (mixed, sources) in enumerate(train_loader):
            # Move data to GPU
            mixed = mixed.to(device)
            sources = sources.to(device)
            
            # Clear previous gradients
            optimizer.zero_grad()
            
            # Forward pass: separate the mixed audio
            estimated_sources = model(mixed)
            
            # Calculate loss
            loss = snr_loss(estimated_sources, sources) # Classic SNR loss
            # loss = si_snr_loss(estimated_sources, sources)  # Alternative
            
            # Backward pass: calculate gradients
            loss.backward()
            
            # Gradient clipping (prevents exploding gradients)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            
            # Update weights
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
            
            # Print progress every 10 batches
            if (batch_idx + 1) % 10 == 0:
                print(f"  Batch {batch_idx + 1}/{len(train_loader)}, Loss: {loss.item():.4f}")
        
        # Calculate average loss for this epoch
        avg_loss = epoch_loss / num_batches
        loss_history.append(avg_loss)
        
        # Update learning rate
        scheduler.step(avg_loss)
        
        # Print epoch summary
        print(f"\nEpoch {epoch + 1}/{config.NUM_EPOCHS}")
        print(f"  Average Loss: {avg_loss:.4f}")
        print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
        print("-" * 40)
        
        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, f'checkpoint_epoch_{epoch + 1}.pth')
            print(f"  Checkpoint saved!")
    
    # # Plot training loss
    # plt.figure(figsize=(10, 5))
    # plt.plot(loss_history)
    # plt.title('Training Loss Over Time')
    # plt.xlabel('Epoch')
    # plt.ylabel('SI-SNR Loss')
    # plt.grid(True)
    # plt.savefig('training_loss.png')
    # plt.show()
    
    return model, loss_history

In [64]:
def train_validate_model(model, train_loader, val_loader, config):
    """
    Train the Conv-TasNet model with validation.
    
    Args:
        model: The Conv-TasNet model
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        config: Configuration object with hyperparameters
    
    Returns:
        model: Trained model (best weights loaded)
        history: Dictionary containing train and validation loss history
    """
    # Move model to GPU if available
    model = model.to(config.DEVICE)
    
    # Set up optimizer
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    
    # Learning rate scheduler (reduces learning rate over time)
    # Reduce learning rate when a metric has stopped improving.
    # Models often benefit from reducing the learning rate by a factor of 2-10 once learning stagnates. 
    # This scheduler reads a metrics quantity and if no improvement is seen for a ‘patience’ number of epochs, the learning rate is reduced.
    
    # Now based on VALIDATION loss instead of training loss (better practice)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.3
    )
    
    # Track losses for plotting
    history = {
        'train_loss': [],
        'val_loss': []
    }
    
    # Early stopping variables
    best_val_loss = float('inf')
    patience = 10  # Stop if no improvement for 10 epochs
    patience_counter = 0
    best_model_state = None
    
    print("\n" + "=" * 60)
    print("Starting Training with Validation")
    print("=" * 60)
    
    for epoch in range(config.NUM_EPOCHS):
        
        # ============================================================
        # TRAINING PHASE
        # ============================================================
        model.train()  # Set model to training mode
        train_loss = 0.0
        num_train_batches = 0
        
        for batch_idx, (mixed, sources) in enumerate(train_loader):
            # Move data to GPU
            mixed = mixed.to(config.DEVICE)
            sources = sources.to(config.DEVICE)
            
            # Clear previous gradients
            optimizer.zero_grad()
            
            # Forward pass: separate the mixed audio
            estimated_sources = model(mixed)
            
            # Calculate loss
            loss = snr_loss(estimated_sources, sources)  # Classic SNR loss
            # loss = si_snr_loss(estimated_sources, sources)  # Alternative
            
            # Backward pass: calculate gradients
            loss.backward()
            
            # Gradient clipping (prevents exploding gradients)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            
            # Update weights
            optimizer.step()
            
            train_loss += loss.item()
            num_train_batches += 1
            
            # Print progress every 10 batches
            if (batch_idx + 1) % 10 == 0:
                print(f"  [Train] Batch {batch_idx + 1}/{len(train_loader)}, "
                      f"Loss: {loss.item():.4f}")
        
        # Calculate average training loss
        avg_train_loss = train_loss / num_train_batches
        history['train_loss'].append(avg_train_loss)
        
        # ============================================================
        # VALIDATION PHASE
        # ============================================================
        model.eval()  # Set model to evaluation mode
        val_loss = 0.0
        num_val_batches = 0
        
        with torch.no_grad():  # Disable gradient computation
            for batch_idx, (mixed, sources) in enumerate(val_loader):
                # Move data to GPU
                mixed = mixed.to(config.DEVICE)
                sources = sources.to(config.DEVICE)
                
                # Forward pass only (no backward pass needed)
                estimated_sources = model(mixed)
                
                # Calculate loss
                loss = snr_loss(estimated_sources, sources)  # Classic SNR loss
                # loss = si_snr_loss(estimated_sources, sources)  # Alternative
                
                val_loss += loss.item()
                num_val_batches += 1
        
        # Calculate average validation loss
        avg_val_loss = val_loss / num_val_batches
        history['val_loss'].append(avg_val_loss)
        
        # ============================================================
        # LEARNING RATE SCHEDULING (based on validation loss)
        # ============================================================
        scheduler.step(avg_val_loss)
        
        # ============================================================
        # EARLY STOPPING & BEST MODEL SAVING
        # ============================================================
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            # Save best model state
            best_model_state = model.state_dict().copy()
            improvement_flag = "✓ New Best!"
        else:
            patience_counter += 1
            improvement_flag = f"✗ No improvement ({patience_counter}/{patience})"
        
        # ============================================================
        # PRINT EPOCH SUMMARY
        # ============================================================
        print(f"\n{'='*60}")
        print(f"Epoch {epoch + 1}/{config.NUM_EPOCHS}")
        print(f"{'='*60}")
        print(f"  Train Loss:      {avg_train_loss:.4f}")
        print(f"  Validation Loss: {avg_val_loss:.4f}")
        print(f"  Best Val Loss:   {best_val_loss:.4f}")
        print(f"  Learning Rate:   {optimizer.param_groups[0]['lr']:.6f}")
        print(f"  Status:          {improvement_flag}")
        print("-" * 60)
        
        # Check early stopping
        if patience_counter >= patience:
            print(f"\n⚠️ Early stopping triggered after {epoch + 1} epochs!")
            print(f"   Best validation loss: {best_val_loss:.4f}")
            break
        
        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': avg_val_loss,
                'best_val_loss': best_val_loss,
            }, f'checkpoint_epoch_{epoch + 1}.pth')
            print(f"  📁 Checkpoint saved!")
    
    # ============================================================
    # LOAD BEST MODEL
    # ============================================================
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n✓ Loaded best model (Val Loss: {best_val_loss:.4f})")
        
        # Save the best model separately
        torch.save({
            'model_state_dict': best_model_state,
            'val_loss': best_val_loss,
        }, 'best_model.pth')
        print("✓ Best model saved to 'best_model.pth'")
    
    return model, history

## Create DataLoaders for Training and Validation

In [65]:
# For training
# Load mixed and source audio
mixed_dir_train = 'D:\Datasets\Source_Separation_Dataset\MiniLibriMix\\train\\mix_clean'
source_dirs_train = ['D:\Datasets\Source_Separation_Dataset\MiniLibriMix\\train\s1', 'D:\Datasets\Source_Separation_Dataset\MiniLibriMix\\train\s2']  # List of source directories
mixed_data_train, source_data_train = load_audio_files(mixed_dir_train, source_dirs_train)

# Create dataset and dataloader
train_dataset = AudioDataset(np.expand_dims(mixed_data_train, axis=1), source_data_train)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)

In [66]:
len(train_loader)

360

In [67]:
# For validation
# Load mixed and source audio
mixed_dir_val = 'D:\Datasets\Source_Separation_Dataset\MiniLibriMix\\val\\mix_clean'
source_dirs_val = ['D:\Datasets\Source_Separation_Dataset\MiniLibriMix\\val\s1', 'D:\Datasets\Source_Separation_Dataset\MiniLibriMix\\val\s2']  # List of source directories
mixed_data_val, source_data_val = load_audio_files(mixed_dir_val, source_dirs_val)

# Create dataset and dataloader
val_dataset = AudioDataset(np.expand_dims(mixed_data_val, axis=1), source_data_val)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

In [68]:
len(val_loader)

21

In [ ]:
# Train with validation!
model = ConvTasNet(config)
trained_model, history = train_model(model, train_loader, val_loader, config)


Starting Training with Validation
  [Train] Batch 10/360, Loss: -1.1037
  [Train] Batch 20/360, Loss: -1.7512
  [Train] Batch 30/360, Loss: -1.8942
  [Train] Batch 40/360, Loss: -2.0677
  [Train] Batch 50/360, Loss: -2.0716
  [Train] Batch 60/360, Loss: -2.1573
  [Train] Batch 70/360, Loss: -2.1209
  [Train] Batch 80/360, Loss: -2.2205
  [Train] Batch 90/360, Loss: -2.2903
  [Train] Batch 100/360, Loss: -2.3857
  [Train] Batch 110/360, Loss: -2.3482
  [Train] Batch 120/360, Loss: -2.0932
  [Train] Batch 130/360, Loss: -2.4036
  [Train] Batch 140/360, Loss: -2.4619
  [Train] Batch 150/360, Loss: -2.4179
  [Train] Batch 160/360, Loss: -2.3170
  [Train] Batch 170/360, Loss: -2.2746
  [Train] Batch 180/360, Loss: -2.4940
  [Train] Batch 190/360, Loss: -2.5286
  [Train] Batch 200/360, Loss: -2.2954
  [Train] Batch 210/360, Loss: -2.4839
  [Train] Batch 220/360, Loss: -2.6391
  [Train] Batch 230/360, Loss: -2.3249
  [Train] Batch 240/360, Loss: -2.5672
  [Train] Batch 250/360, Loss: -2.4157

## 8. Testing/Inference Function

In [70]:
mixed_signal, _ = librosa.load('14-212-0013_7926-254949-0054.wav', sr=config.SAMPLING_RATE)
mixed_signal.shape

(60560,)

In [71]:
Audio(mixed_signal, rate=config.SAMPLING_RATE)

In [72]:
def create_windows_gen(signal, config):
    """
    Create overlapping windows with Hamming window.
    
    Args:
        signal: Input audio signal (1D numpy array)
        window_size: Size of each window
        hop_size: Hop size between windows
        
    Returns:
        windows: List of windowed signal segments
        num_windows: Number of windows created
    """
    windows = []
    hamming_window = np.hamming(config.WINDOW_SIZE)
    
    # Pad signal if necessary to ensure we cover the entire audio
    pad_length = (config.WINDOW_SIZE - len(signal) % config.HOP_SIZE) % config.HOP_SIZE
    if pad_length > 0:
        signal = np.pad(signal, (0, pad_length), mode='constant')
    
    # Also pad to ensure last samples are covered
    signal = np.pad(signal, (0, config.WINDOW_SIZE), mode='constant')
    
    for start in range(0, len(signal) - config.WINDOW_SIZE + 1, config.HOP_SIZE):
        window = signal[start:start + config.WINDOW_SIZE]
        windowed_signal = window * hamming_window
        windows.append(windowed_signal)
    
    return np.array(windows)

In [73]:
windows = create_windows_gen(mixed_signal, config)
windows.shape

(17, 32000)

In [75]:
def overlap_add(windows, original_length, config):
    """
    Reconstructs a signal from overlapping windows using the overlap-add method.
    
    Args:
        windows (numpy.ndarray): 2D array of shape (num_windows, window_size).
        original_length (int): The length of the original signal before padding
        hop_size (int): The hop size used for creating the windows.

    Returns:
        numpy.ndarray: The reconstructed signal.
    """
    # Initialize the output signal
    output_length = (windows.shape[0] - 1) * config.HOP_SIZE + windows.shape[1]
    output_signal = np.zeros(output_length)
    
    # Overlap-Add
    for i, window in enumerate(windows):
        start = i * config.HOP_SIZE
        output_signal[start:start + len(window)] += window
        
    output_signal = output_signal[:original_length]
    return output_signal

In [76]:
original_length = mixed_signal.shape[0]

In [77]:
output_signal = overlap_add(windows, original_length, config)

In [78]:
# Play the audio
Audio(output_signal, rate=SAMPLING_RATE)

In [79]:
def overlap_add_sources(windows, original_length, config):
    """
    Reconstructs signals from overlapping windows using the overlap-add method.
    
    Args:
        windows (numpy.ndarray): 3D array of shape (num_windows, num_sources, window_length).
        original_length (int): The length of the original signal to be reconstructed.
        hop_size (int): The hop size used for creating the windows.

    Returns:
        numpy.ndarray: The reconstructed signals of shape (num_sources, original_length).
    """
    num_windows, num_sources, window_length = windows.shape
    
    # Initialize the output signal for each source
    output_length = (num_windows - 1) * config.HOP_SIZE + window_length
    output_signals = np.zeros((num_sources, output_length))
    
    # Overlap-Add for each source
    for i in range(num_windows):
        start = i * config.HOP_SIZE
        for source in range(num_sources):
            output_signals[source, start:start + window_length] += windows[i, source]
    
    # Trim to the original length
    output_signals = output_signals[:, :original_length]
    return output_signals

In [80]:
def separate_audio(model, audio_path, config):
    """
    Use the trained model to separate an audio file.
    """
    model.eval()  # Set model to evaluation mode
    
    # Load audio
    mixed_signal, _ = librosa.load(audio_path, sr=config.SAMPLING_RATE)
    original_length = mixed_signal.size
    
    # create windows
    windows = create_windows_gen(mixed_signal, config)
    input_tensor = torch.FloatTensor(windows).unsqueeze(1).to(config.DEVICE)   
    
    # Perform inference    
    with torch.no_grad():        
        estimated_sources = model(input_tensor)  
        # Shape: (num_windows, num_sources, window_length)    
        separated_windows = estimated_sources.cpu().numpy()
    
    separated_signal = overlap_add_sources(separated_windows, original_length, config)
    
    return separated_signal  # Return (num_sources, audio_length)

In [81]:
model = ConvTasNet(config).to(config.DEVICE)

In [82]:
best_model = torch.load('best_model.pth')

In [83]:
model.load_state_dict(best_model['model_state_dict'])

<All keys matched successfully>

In [84]:
separated_signal = separate_audio(model, '14-212-0013_7926-254949-0054.wav', config)

In [85]:
separated_signal.shape

(2, 60560)

In [ ]:
### Play the audio
Audio(separated_signal[0,:], rate=SAMPLING_RATE)

## 9. Main Function - Put it all together

In [ ]:
def main():
    """
    Main function to run the complete pipeline.
    """
    print("=" * 60)
    print("Conv-TasNet Audio Source Separation")
    print("=" * 60)
    print(f"\nDevice: {device}")
    print(f"Number of sources: {config.NUM_SOURCES}")
    
    # Create the model
    model = ConvTasNet(config)
    
    # Count parameters
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}")
    
    # ---- OPTION 1: Train with your own data ----
    # Uncomment and modify the paths below to train:
    
    # mixed_dir = 'path/to/mixed/audio'
    # source_dirs = ['path/to/source1', 'path/to/source2']
    # 
    # dataset = AudioDataset(mixed_dir, source_dirs, config)
    # train_loader = DataLoader(
    #     dataset, 
    #     batch_size=config.BATCH_SIZE, 
    #     shuffle=True,
    #     num_workers=2
    # )
    # 
    # trained_model, loss_history = train_model(model, train_loader, config, device)
    # 
    # # Save the final model
    # torch.save(trained_model.state_dict(), 'conv_tasnet_final.pth')
    
    # ---- OPTION 2: Test with dummy data ----
    print("\n" + "=" * 60)
    print("Testing with dummy data")
    print("=" * 60)
    
    # Create dummy mixed audio (2 seconds)
    test_length = 32000
    dummy_mixed = torch.randn(1, 1, test_length).to(device)
    
    model = model.to(device)
    model.eval()
    
    with torch.no_grad():
        output = model(dummy_mixed)
    
    print(f"\nInput shape:  {dummy_mixed.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Expected: (1, {config.NUM_SOURCES}, ~{test_length})")
    print("\nModel is working correctly!")
    
    # Visualize the test
    fig, axes = plt.subplots(3, 1, figsize=(12, 8))
    
    # Plot input
    axes[0].plot(dummy_mixed[0, 0].cpu().numpy())
    axes[0].set_title('Input: Mixed Audio')
    axes[0].set_xlabel('Samples')
    axes[0].set_ylabel('Amplitude')
    
    # Plot outputs
    for i in range(min(config.NUM_SOURCES, 2)):
        axes[i + 1].plot(output[0, i].cpu().numpy())
        axes[i + 1].set_title(f'Output: Separated Source {i + 1}')
        axes[i + 1].set_xlabel('Samples')
        axes[i + 1].set_ylabel('Amplitude')
    
    plt.tight_layout()
    plt.savefig('separation_test.png')
    plt.show()
    
    print("\nTest visualization saved as 'separation_test.png'")

# Run the main function
if __name__ == "__main__":
    main()


In [ ]:
 # ============================================================
    # PLOT TRAINING & VALIDATION LOSS
    # ============================================================
    plt.figure(figsize=(12, 5))
    
    # Loss plot
    plt.subplot(1, 2, 1)
    epochs_range = range(1, len(history['train_loss']) + 1)
    plt.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    plt.plot(epochs_range, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('SNR Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Gap between train and val (overfitting indicator)
    plt.subplot(1, 2, 2)
    gap = [t - v for t, v in zip(history['train_loss'], history['val_loss'])]
    plt.plot(epochs_range, gap, 'g-', linewidth=2)
    plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    plt.xlabel('Epoch')
    plt.ylabel('Train Loss - Val Loss')
    plt.title('Generalization Gap (Overfitting Indicator)')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_validation_loss.png', dpi=150)
    plt.show()